<a href="https://colab.research.google.com/github/dmainagithub/LLMs-Lessons/blob/main/huggingface_text_classification_tutorial.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Text Classification Tutorial

Note: a GPU is needed in google colab: Runtime -> Change runtime type -> Hardware accelerator -> GPU

## 2. Import necessary commands

In [70]:
# Install dependencies
try:
  import datasets, evaluate, accelerate
  import gradio as gr
except ModuleNotFoundError:
  !pip install -U datasets evaluate accelerate gradio # -U stands for upgrade
  import datasets, evaluate, accelerate
  import gradio as gr

import random

import numpy as np
import pandas as pd

import torch
import transformers

print(f"Using transformers version: {transformers.__version__}")
print(f"Using datasets version: {datasets.__version__}")
print(f"Using torch version: {torch.__version__}")



Using transformers version: 5.16.1
Using datasets version: 4.0.0
Using torch version: 2.11.0+cu128


## 3. Getting a dataset

In [71]:
from datasets import load_dataset

dataset = load_dataset(path="mrdbourke/learn_hf_food_not_food_image_captions")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 250
    })
})

In [72]:
# What features are there
dataset.column_names

{'train': ['text', 'label']}

In [73]:
# Access the training split
dataset["train"]

Dataset({
    features: ['text', 'label'],
    num_rows: 250
})

In [74]:
dataset["train"][0]

{'text': 'Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
 'label': 'food'}

### Inspect random samples

In [75]:
import random

random_indexs = random.sample(range(len(dataset["train"])), 5)
print(random_indexs)

random_samples = dataset["train"][random_indexs]

print(f"[INFO] Random samples from dataset:\n")
for text, label in zip(random_samples["text"], random_samples["label"]):
  print(f" Text: {text} | Label: {label}")


[79, 173, 58, 115, 55]
[INFO] Random samples from dataset:

 Text: Creamy spinach and potato curry, featuring fluffy potatoes and nutritious spinach in a rich sauce with cream and garam masala. | Label: food
 Text: Microscope set up on a table | Label: not_food
 Text: A boy building a fort in the living room with his curious cat watching | Label: not_food
 Text: Sushi platter showcasing a variety of colorful rolls and garnishes. | Label: food
 Text: Uniquely shaped sushi roll, such as a heart or flower. | Label: food


In [76]:
dataset["train"].unique("label")

['food', 'not_food']

In [77]:
# Check the count of each label
from collections import Counter

Counter(dataset["train"]["label"])


Counter({'food': 125, 'not_food': 125})

In [78]:
# Turn our dataset into a dataframe
food_not_food_df = pd.DataFrame(dataset["train"])
food_not_food_df.sample(7)

,text,label
172,"Spicy prawn curry with fresh mint garnish, fea...",food
31,"Potatoes, onions, garlic, cauliflower, and bro...",food
185,"Vibrant red curry with tofu and bell peppers, ...",food
25,Mailbox standing by a front door,not_food
98,Set of cake pans tucked in a drawer,not_food
4,Lawn mower stored in a shed,not_food
215,Spicy tuna sushi roll with a crispy tempura co...,food


In [79]:
food_not_food_df["label"].value_counts()

,count
label,
food,125
not_food,125


## 4. Preparing data for text classification

1. Tokenization (machines prefer numbers rather than words)

2. Creating a train-test split (train split for training and test split for evaluation)

In [80]:
# Create a mapping programmatically
id2label = {idx: label for idx, label in enumerate(dataset["train"].unique("label")[::-1])}
label2id = {label: idx for idx, label in id2label.items()}

print(id2label)
print(label2id)

{0: 'not_food', 1: 'food'}
{'not_food': 0, 'food': 1}


In [81]:
id2label = {}
for idx, label in enumerate(dataset["train"].unique("label")[::-1]):
  print(idx, label)
  id2label[idx] = label

0 not_food
1 food


In [82]:
# Turn labels into 0 or 1
def map_labels_to_number(example):
  example["label"] = label2id[example["label"]]
  return example

example_sample ={"text": "This is a sentence about my favorite food: honey", "label": "food"}

# Test our function
map_labels_to_number(example_sample)

{'text': 'This is a sentence about my favorite food: honey', 'label': 1}

In [83]:
# Map our dataset labels to numbers (the whole dataset)
# With dataset.map()
dataset = dataset["train"].map(map_labels_to_number)
dataset[:5]

Map:   0%|          | 0/250 [00:00<?, ? examples/s]

{'text': ['Creamy cauliflower curry with garlic naan, featuring tender cauliflower in a rich sauce with cream and spices, served with garlic naan bread.',
  'Set of books stacked on a desk',
  'Watching TV together, a family has their dog stretched out on the floor',
  'Wooden dresser with a mirror reflecting the room',
  'Lawn mower stored in a shed'],
 'label': [1, 0, 0, 0, 0]}